# Plant Disease Detection - Data Preprocessing & EDA

**Author:** Pratham Rajesh & Shreram Palanisamy  
**Date:** November 2025  
**Purpose:** Data understanding, preprocessing, and exploratory data analysis

## CRISP-DM Phase 2: Data Understanding & Phase 3: Data Preparation

This notebook covers:
1. Dataset loading and exploration
2. Class distribution analysis
3. Image quality assessment
4. Train/Val/Test splits (70/15/15 stratified)
5. Data augmentation examples
6. Creating stratified samples for ablation studies

## 1. Setup Environment

In [ ]:
# Mount Google Drive (Colab only)
try:
    from google.colab import drive
    drive.mount('/content/drive')
    IN_COLAB = True
    print("Running in Google Colab")
except:
    IN_COLAB = False
    print("Running locally")

In [ ]:
# Install required packages (Colab only)
if IN_COLAB:
    !pip install -q albumentations opencv-python-headless scikit-learn tqdm

In [ ]:
# Imports
import os
import json
import shutil
from pathlib import Path
from collections import Counter
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image
from tqdm.auto import tqdm
from sklearn.model_selection import train_test_split
import albumentations as A
from albumentations.pytorch import ToTensorV2

# Set random seeds for reproducibility
RANDOM_SEED = 42
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

# Plotting settings
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")
%matplotlib inline

print("✓ All imports successful")

## 2. Dataset Paths Configuration

In [ ]:
# Configure paths based on environment
if IN_COLAB:
    # Google Colab paths
    RAW_DATA_PATH = '/content/drive/MyDrive/plant_disease_data/Plant_leave_diseases_dataset_without_augmentation'
    PROCESSED_DATA_PATH = '/content/drive/MyDrive/plant_disease_data/processed'
    RESULTS_PATH = '/content/drive/MyDrive/plant_disease_data/results'
else:
    # Local paths
    BASE_PATH = Path('/Users/prathamr/Documents/plant-disease-detection')
    RAW_DATA_PATH = BASE_PATH / 'Plant_leave_diseases_dataset_without_augmentation'
    PROCESSED_DATA_PATH = BASE_PATH / 'data' / 'processed'
    RESULTS_PATH = BASE_PATH / 'results'

# Create output directories
Path(PROCESSED_DATA_PATH).mkdir(parents=True, exist_ok=True)
Path(RESULTS_PATH).mkdir(parents=True, exist_ok=True)

print(f"Raw data path: {RAW_DATA_PATH}")
print(f"Processed data path: {PROCESSED_DATA_PATH}")
print(f"Results path: {RESULTS_PATH}")

## 3. Dataset Exploration

In [ ]:
# Get all class directories
class_dirs = sorted([d for d in os.listdir(RAW_DATA_PATH) if os.path.isdir(os.path.join(RAW_DATA_PATH, d))])

print(f"Total number of classes: {len(class_dirs)}")
print("\nClass names:")
for i, class_name in enumerate(class_dirs, 1):
    print(f"{i:2d}. {class_name}")

In [ ]:
# Count images per class
class_counts = {}
total_images = 0

for class_name in tqdm(class_dirs, desc="Counting images"):
    class_path = os.path.join(RAW_DATA_PATH, class_name)
    images = [f for f in os.listdir(class_path) if f.lower().endswith(('.jpg', '.jpeg', '.png'))]
    class_counts[class_name] = len(images)
    total_images += len(images)

print(f"\nTotal images in dataset: {total_images:,}")
print(f"Minimum images per class: {min(class_counts.values())}")
print(f"Maximum images per class: {max(class_counts.values())}")
print(f"Average images per class: {total_images / len(class_dirs):.0f}")

## 4. Visualization 1: Class Distribution

In [ ]:
# Create class distribution plot
fig, ax = plt.subplots(figsize=(14, 10))

# Sort by count
sorted_classes = sorted(class_counts.items(), key=lambda x: x[1], reverse=True)
classes, counts = zip(*sorted_classes)

# Create horizontal bar chart
y_pos = np.arange(len(classes))
colors = plt.cm.viridis(np.linspace(0, 1, len(classes)))

bars = ax.barh(y_pos, counts, color=colors)
ax.set_yticks(y_pos)
ax.set_yticklabels(classes, fontsize=9)
ax.invert_yaxis()
ax.set_xlabel('Number of Images', fontsize=12, fontweight='bold')
ax.set_title('Image Distribution Across 39 Plant Disease Classes', fontsize=14, fontweight='bold', pad=20)
ax.grid(axis='x', alpha=0.3)

# Add value labels on bars
for i, (bar, count) in enumerate(zip(bars, counts)):
    ax.text(count + 50, i, str(count), va='center', fontsize=8)

plt.tight_layout()
plt.savefig(os.path.join(RESULTS_PATH, 'class_distribution.png'), dpi=300, bbox_inches='tight')
plt.show()

print("✓ Visualization 1/12 complete: Class Distribution")

## 5. Visualization 2: Sample Images from Each Class

In [ ]:
# Display sample images from random classes
num_classes_to_show = 8
sample_classes = random.sample(class_dirs, num_classes_to_show)

fig, axes = plt.subplots(2, 4, figsize=(16, 8))
axes = axes.ravel()

for idx, class_name in enumerate(sample_classes):
    class_path = os.path.join(RAW_DATA_PATH, class_name)
    images = [f for f in os.listdir(class_path) if f.lower().endswith(('.jpg', '.jpeg', '.png'))]
    sample_img_path = os.path.join(class_path, random.choice(images))

    img = Image.open(sample_img_path).convert('RGB')
    axes[idx].imshow(img)
    axes[idx].set_title(class_name.replace('___', '\n'), fontsize=10, fontweight='bold')
    axes[idx].axis('off')

plt.suptitle('Sample Images from Different Disease Classes', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_PATH, 'sample_images.png'), dpi=300, bbox_inches='tight')
plt.show()

print("✓ Visualization 2/12 complete: Sample Images")

## 6. Create Class Mapping

In [ ]:
# Create class index to name mapping
class_to_idx = {class_name: idx for idx, class_name in enumerate(sorted(class_dirs))}
idx_to_class = {idx: class_name for class_name, idx in class_to_idx.items()}

# Save mapping to JSON
mapping_data = {
    'class_to_idx': class_to_idx,
    'idx_to_class': {str(k): v for k, v in idx_to_class.items()},  # JSON requires string keys
    'num_classes': len(class_dirs),
    'class_names': sorted(class_dirs)
}

mapping_path = os.path.join(PROCESSED_DATA_PATH, 'class_mapping.json')
with open(mapping_path, 'w') as f:
    json.dump(mapping_data, f, indent=2)

print(f"✓ Class mapping saved to {mapping_path}")
print(f"  Number of classes: {len(class_dirs)}")

## 7. Collect All Image Paths and Labels

In [ ]:
# Collect all image paths and corresponding labels
all_image_paths = []
all_labels = []

for class_name in tqdm(sorted(class_dirs), desc="Collecting image paths"):
    class_path = os.path.join(RAW_DATA_PATH, class_name)
    class_idx = class_to_idx[class_name]

    for img_name in os.listdir(class_path):
        if img_name.lower().endswith(('.jpg', '.jpeg', '.png')):
            img_path = os.path.join(class_path, img_name)
            all_image_paths.append(img_path)
            all_labels.append(class_idx)

print(f"\nCollected {len(all_image_paths):,} images with labels")
print(f"Label distribution: {Counter(all_labels).most_common(5)}... (showing top 5)")

## 8. Train/Val/Test Split (70/15/15 Stratified)

In [ ]:
# First split: 70% train, 30% temp
train_paths, temp_paths, train_labels, temp_labels = train_test_split(
    all_image_paths,
    all_labels,
    test_size=0.3,
    stratify=all_labels,
    random_state=RANDOM_SEED
)

# Second split: 15% val, 15% test (from 30% temp)
val_paths, test_paths, val_labels, test_labels = train_test_split(
    temp_paths,
    temp_labels,
    test_size=0.5,
    stratify=temp_labels,
    random_state=RANDOM_SEED
)

print("Dataset Split:")
print(f"  Train: {len(train_paths):,} images ({len(train_paths)/len(all_image_paths)*100:.1f}%)")
print(f"  Val:   {len(val_paths):,} images ({len(val_paths)/len(all_image_paths)*100:.1f}%)")
print(f"  Test:  {len(test_paths):,} images ({len(test_paths)/len(all_image_paths)*100:.1f}%)")
print(f"  Total: {len(all_image_paths):,} images")

In [ ]:
# Verify stratification - check class distribution in each split
train_dist = Counter(train_labels)
val_dist = Counter(val_labels)
test_dist = Counter(test_labels)

# Create comparison DataFrame
split_comparison = pd.DataFrame({
    'Class': [idx_to_class[i] for i in range(len(class_dirs))],
    'Train': [train_dist.get(i, 0) for i in range(len(class_dirs))],
    'Val': [val_dist.get(i, 0) for i in range(len(class_dirs))],
    'Test': [test_dist.get(i, 0) for i in range(len(class_dirs))]
})

split_comparison['Total'] = split_comparison[['Train', 'Val', 'Test']].sum(axis=1)
split_comparison['Train%'] = (split_comparison['Train'] / split_comparison['Total'] * 100).round(1)
split_comparison['Val%'] = (split_comparison['Val'] / split_comparison['Total'] * 100).round(1)
split_comparison['Test%'] = (split_comparison['Test'] / split_comparison['Total'] * 100).round(1)

print("\nSplit verification (first 5 classes):")
print(split_comparison.head())

# Save split information
split_comparison.to_csv(os.path.join(RESULTS_PATH, 'split_distribution.csv'), index=False)
print(f"\n✓ Split distribution saved to {RESULTS_PATH}/split_distribution.csv")

## 9. Save Splits to CSV Files

In [ ]:
# Save train/val/test splits to CSV files
def save_split_to_csv(paths, labels, split_name, output_dir):
    """Save image paths and labels to CSV file."""
    df = pd.DataFrame({
        'image_path': paths,
        'label': labels,
        'class_name': [idx_to_class[label] for label in labels]
    })
    csv_path = os.path.join(output_dir, f'{split_name}.csv')
    df.to_csv(csv_path, index=False)
    print(f"✓ Saved {split_name} split to {csv_path}")
    return df

# Save all splits
train_df = save_split_to_csv(train_paths, train_labels, 'train', PROCESSED_DATA_PATH)
val_df = save_split_to_csv(val_paths, val_labels, 'val', PROCESSED_DATA_PATH)
test_df = save_split_to_csv(test_paths, test_labels, 'test', PROCESSED_DATA_PATH)

print(f"\n✓ All splits saved to {PROCESSED_DATA_PATH}")

## 10. Visualization 3: Data Augmentation Examples

In [ ]:
# Define augmentation pipeline
augmentation = A.Compose([
    A.Resize(224, 224),
    A.HorizontalFlip(p=1.0),
    A.VerticalFlip(p=0.0),
    A.Rotate(limit=30, p=1.0),
    A.RandomBrightnessContrast(brightness_limit=0.2, contrast_limit=0.2, p=1.0),
    A.ColorJitter(p=1.0),
])

# Select a sample image
sample_img_path = random.choice(train_paths)
sample_img = np.array(Image.open(sample_img_path).convert('RGB'))

# Create augmented versions
fig, axes = plt.subplots(2, 4, figsize=(16, 8))
axes = axes.ravel()

# Original image
axes[0].imshow(sample_img)
axes[0].set_title('Original Image', fontsize=12, fontweight='bold')
axes[0].axis('off')

# Augmented versions
augmentation_names = [
    'Horizontal Flip',
    'Rotation',
    'Brightness/Contrast',
    'Color Jitter',
    'Combined 1',
    'Combined 2'
]

for idx in range(1, 7):
    augmented = augmentation(image=sample_img)['image']
    axes[idx].imshow(augmented)
    axes[idx].set_title(f'Augmentation {idx}', fontsize=12, fontweight='bold')
    axes[idx].axis('off')

axes[7].axis('off')  # Hide last subplot

plt.suptitle('Data Augmentation Examples', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_PATH, 'augmentation_examples.png'), dpi=300, bbox_inches='tight')
plt.show()

print("✓ Visualization 3/12 complete: Augmentation Examples")

## 11. Create Stratified Samples for Ablation Studies

In [ ]:
# Create 25% and 50% stratified samples for faster ablation experiments
def create_stratified_sample(paths, labels, sample_size, output_name):
    """Create stratified sample and save to CSV."""
    sample_paths, _, sample_labels, _ = train_test_split(
        paths,
        labels,
        train_size=sample_size,
        stratify=labels,
        random_state=RANDOM_SEED
    )

    df = pd.DataFrame({
        'image_path': sample_paths,
        'label': sample_labels,
        'class_name': [idx_to_class[label] for label in sample_labels]
    })

    # Save to CSV
    sample_dir = os.path.join(PROCESSED_DATA_PATH, f'stratified_sample_{int(sample_size*100)}pct')
    os.makedirs(sample_dir, exist_ok=True)
    csv_path = os.path.join(sample_dir, f'{output_name}.csv')
    df.to_csv(csv_path, index=False)

    print(f"✓ Created {sample_size*100:.0f}% sample: {len(sample_paths):,} images")
    return df

# Create samples
sample_25_train = create_stratified_sample(train_paths, train_labels, 0.25, 'train')
sample_50_train = create_stratified_sample(train_paths, train_labels, 0.50, 'train')

# Also create smaller val/test sets for these samples
sample_25_val = create_stratified_sample(val_paths, val_labels, 0.25, 'val')
sample_50_val = create_stratified_sample(val_paths, val_labels, 0.50, 'val')

print("\n✓ Stratified samples created for ablation studies")

## 12. Summary Statistics

In [ ]:
# Create comprehensive summary
summary = {
    'dataset_name': 'PlantVillage',
    'total_images': len(all_image_paths),
    'num_classes': len(class_dirs),
    'min_images_per_class': min(class_counts.values()),
    'max_images_per_class': max(class_counts.values()),
    'avg_images_per_class': len(all_image_paths) / len(class_dirs),
    'train_images': len(train_paths),
    'val_images': len(val_paths),
    'test_images': len(test_paths),
    'train_split': 0.70,
    'val_split': 0.15,
    'test_split': 0.15,
    'random_seed': RANDOM_SEED,
    'stratified': True
}

# Save summary
summary_path = os.path.join(PROCESSED_DATA_PATH, 'dataset_summary.json')
with open(summary_path, 'w') as f:
    json.dump(summary, f, indent=2)

print("Dataset Summary:")
print("=" * 50)
for key, value in summary.items():
    print(f"{key:25s}: {value}")
print("=" * 50)
print(f"\n✓ Summary saved to {summary_path}")

## 13. Notebook Complete

### Outputs Created:
1. ✅ Class mapping JSON file
2. ✅ Train/Val/Test CSV files (70/15/15 stratified split)
3. ✅ Stratified samples (25% and 50%) for ablation studies
4. ✅ Visualization 1: Class distribution plot
5. ✅ Visualization 2: Sample images from classes
6. ✅ Visualization 3: Data augmentation examples
7. ✅ Split distribution CSV
8. ✅ Dataset summary JSON

### Next Steps:
- **Notebook 02**: Model training with all architectures and ablation studies
- **Notebook 03**: Evaluation and comprehensive visualizations
- **Notebook 04**: AutoGluon comparison

In [ ]:
print("\n" + "=" * 60)
print("NOTEBOOK 01 COMPLETE")
print("=" * 60)
print("\nData preprocessing and EDA successfully completed!")
print(f"All outputs saved to: {PROCESSED_DATA_PATH}")
print(f"All visualizations saved to: {RESULTS_PATH}")
print("\nReady to proceed to Model Training (Notebook 02)")